# SpillTrace — Evaluation Baseline

This notebook provides the Day 3 evaluation template for SpillTrace.

Evaluation areas:
1. Segmentation metrics
2. Geometry checks
3. Drift tests
4. AIS quality
5. Candidate ranking

Metrics are explicitly classified as measured/available or unavailable.
No unavailable ground-truth metric is estimated or fabricated.

## 1. Evaluation Status

This evaluation distinguishes between:

- Real/measured metrics: calculated from available project outputs or real observations.
- Controlled validation metrics: calculated using the explicitly provided controlled simulation.
- Unavailable metrics: metrics requiring ground truth or reference data that is not available.

Controlled synthetic results must not be interpreted as real-world oceanographic accuracy.

## 2. Segmentation Metrics

Metrics:
- IoU
- Dice Score
- Precision
- Recall

Source:
Existing segmentation evaluation notebook (`02_segmentation_metrics.ipynb`).

Status:
Measured on the available evaluation example.

Note:
The available example is a metric-function validation example, not a claim of model performance on a real ground-truth dataset.

## 3. Geometry Checks

Geometry metrics:
- Slick area
- Centroid
- Perimeter

Source:
`analytics/geometry_metrics.py`
and `03_baseline_metrics.ipynb`.

Status:
Measured against known-value geometry validation.

Validated values:
- Area: 400 m²
- Centroid row: 0.5
- Centroid column: 0.5
- Perimeter: 80 m

## 4. Drift Tests

Two evaluation scenarios are considered.

### 4.1 Controlled Synthetic Simulation — Drift Engine Validation Only

Scenario:
`drift_validation_sim_001`

Known source:
- Latitude: 19.5
- Longitude: 70.5

Predicted origin:
- Latitude: 19.499943
- Longitude: 70.5

Provided origin error:
- 6.2596 m

Purpose:
Validate the drift-engine implementation only.

This controlled simulation does not validate real-world oceanographic accuracy.

In [ ]:
from geopy.distance import geodesic

known_source = (19.5, 70.5)
predicted_origin = (19.499943, 70.5)

calculated_error_m = geodesic(
    known_source,
    predicted_origin
).meters

provided_error_m = 6.2596

difference_m = calculated_error_m - provided_error_m
absolute_difference_m = abs(difference_m)

print(f"Calculated geodesic error: {calculated_error_m:.4f} m")
print(f"Provided error: {provided_error_m:.4f} m")
print(f"Absolute difference: {absolute_difference_m:.4f} m")

### 4.2 Real SAR Observation with Analyst Parameter-Driven Drift Simulation

Scenario:
`SPILL_TEST3_001`

Predicted origin:
- Latitude: 19.495748
- Longitude: 70.709029

Actual/reference origin:
- Unavailable

Therefore:
- Predicted-vs-actual origin error: UNAVAILABLE
- Real-world origin accuracy: NOT EVALUATED

Available drift evaluation:
- Hindcast path validity
- Backward direction consistency
- Uncertainty polygon validity
- Final uncertainty radius
- Reproducibility
- Wind/current parameter sensitivity
- Runtime
- Predicted origin inside uncertainty polygon
- GeoJSON geometry validity

Final uncertainty radius:
1191.87 m

## 4.3 Hindcast Path and Uncertainty Geometry Validation

### Hindcast Path

- Geometry type: LineString
- Number of path points: 13
- Geometry validity: Valid
- Path is non-empty: Yes
- Longitude direction: Monotonically decreasing
- Starting point: (71.058397, 19.495817)
- Final point: (70.709029, 19.495748)
- Predicted origin: (70.709029, 19.495748)
- Final path point matches predicted origin: Yes

### Backward Direction Consistency

The hindcast path progresses consistently toward the predicted origin, with longitude decreasing from the observed slick-side location toward the predicted origin.

Status: PASS

### Uncertainty Polygon

- Geometry type: Polygon
- Geometry validity: Valid
- Polygon is non-empty: Yes
- Predicted origin lies inside polygon: Yes
- Final uncertainty radius: 1191.87 m

Status: PASS

### Interpretation

These checks validate the structural and geometric consistency of the provided drift output. They do not establish real-world oceanographic accuracy or verified origin-location accuracy.

##  Drift Evaluation

The drift evaluation uses two categories:

1. Controlled Synthetic Simulation — Drift Engine Validation Only
2. Real SAR Observation with Analyst Parameter-Driven Drift Simulation

The real SAR scenario does not have independent ground-truth origin coordinates.
Therefore, predicted-vs-actual origin error is not reported for the real scenario.

In [1]:
import pandas as pd
from math import radians, sin, cos, asin, sqrt

drift_results = [
    {
        "scenario": "Baseline Run",
        "wind_speed_mps": 7.7167,
        "current_speed_mps": 0.6173,
        "random_seed": 42,
        "execution_time_s": 0.7586,
        "origin_lat": 19.495748,
        "origin_lon": 70.709029,
        "uncertainty_radius_m": 1191.87
    },
    {
        "scenario": "Reproducibility Check (Same Seed)",
        "wind_speed_mps": 7.7167,
        "current_speed_mps": 0.6173,
        "random_seed": 42,
        "execution_time_s": 0.7626,
        "origin_lat": 19.495748,
        "origin_lon": 70.709029,
        "uncertainty_radius_m": 1191.87
    },
    {
        "scenario": "Sensitivity: High Wind (+20%)",
        "wind_speed_mps": 9.26004,
        "current_speed_mps": 0.6173,
        "random_seed": 42,
        "execution_time_s": 0.7901,
        "origin_lat": 19.495745,
        "origin_lon": 70.689975,
        "uncertainty_radius_m": 1191.87
    },
    {
        "scenario": "Sensitivity: High Current (+20%)",
        "wind_speed_mps": 7.7167,
        "current_speed_mps": 0.74076,
        "random_seed": 42,
        "execution_time_s": 0.7340,
        "origin_lat": 19.495739,
        "origin_lon": 70.658223,
        "uncertainty_radius_m": 1191.88
    },
    {
        "scenario": "Seed Variance (Seed 99)",
        "wind_speed_mps": 7.7167,
        "current_speed_mps": 0.6173,
        "random_seed": 99,
        "execution_time_s": 0.7759,
        "origin_lat": 19.496178,
        "origin_lon": 70.709362,
        "uncertainty_radius_m": 1149.12
    }
]

df_drift = pd.DataFrame(drift_results)
df_drift

,scenario,wind_speed_mps,current_speed_mps,random_seed,execution_time_s,origin_lat,origin_lon,uncertainty_radius_m
0,Baseline Run,7.71670,0.61730,42,0.7586,19.495748,70.709029,1191.87
1,Reproducibility Check (Same Seed),7.71670,0.61730,42,0.7626,19.495748,70.709029,1191.87
2,Sensitivity: High Wind (+20%),9.26004,0.61730,42,0.7901,19.495745,70.689975,1191.87
3,Sensitivity: High Current (+20%),7.71670,0.74076,42,0.7340,19.495739,70.658223,1191.88
4,Seed Variance (Seed 99),7.71670,0.61730,99,0.7759,19.496178,70.709362,1149.12


In [2]:
baseline = df_drift.iloc[0]
same_seed = df_drift.iloc[1]

reproducible = (
    baseline["origin_lat"] == same_seed["origin_lat"]
    and baseline["origin_lon"] == same_seed["origin_lon"]
    and baseline["uncertainty_radius_m"] == same_seed["uncertainty_radius_m"]
)

print("Reproducible with seed=42:", reproducible)
print("Baseline execution time:", baseline["execution_time_s"], "s")
print("Repeat execution time:", same_seed["execution_time_s"], "s")

Reproducible with seed=42: True
Baseline execution time: 0.7586 s
Repeat execution time: 0.7626 s


In [3]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000

    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = (
        sin(dlat / 2) ** 2
        + cos(radians(lat1))
        * cos(radians(lat2))
        * sin(dlon / 2) ** 2
    )

    return 2 * R * asin(sqrt(a))


baseline_lat = baseline["origin_lat"]
baseline_lon = baseline["origin_lon"]

for i in [2, 3, 4]:
    row = df_drift.iloc[i]

    displacement = haversine_m(
        baseline_lat,
        baseline_lon,
        row["origin_lat"],
        row["origin_lon"]
    )

    print(f"{row['scenario']}: {displacement:.2f} m from baseline")

Sensitivity: High Wind (+20%): 1997.23 m from baseline
Sensitivity: High Current (+20%): 5325.47 m from baseline
Seed Variance (Seed 99): 59.20 m from baseline


In [4]:
print("Runtime summary:")
print(df_drift[["scenario", "execution_time_s"]].to_string(index=False))

Runtime summary:
                         scenario  execution_time_s
                     Baseline Run            0.7586
Reproducibility Check (Same Seed)            0.7626
    Sensitivity: High Wind (+20%)            0.7901
 Sensitivity: High Current (+20%)            0.7340
          Seed Variance (Seed 99)            0.7759


### Final Uncertainty Radius

For the selected baseline real SAR drift run:

- Final uncertainty radius = 1191.87 m
- Evaluation status = measured from provided drift output

This value represents the modelled uncertainty radius and must not be interpreted as verified real-world origin accuracy.

In [5]:
known_source = (19.5, 70.5)
predicted_origin = (19.499943, 70.5)

calculated_error_m = haversine_m(
    known_source[0],
    known_source[1],
    predicted_origin[0],
    predicted_origin[1]
)

provided_error_m = 6.2596

print(f"Calculated error: {calculated_error_m:.4f} m")
print(f"Provided error: {provided_error_m:.4f} m")
print(f"Difference: {abs(calculated_error_m - provided_error_m):.4f} m")

Calculated error: 6.3381 m
Provided error: 6.2596 m
Difference: 0.0785 m


### Drift Evaluation Conclusion

- Same-seed runs produced identical predicted origin coordinates and uncertainty radius, confirming reproducibility for seed=42.
- Increasing wind speed by 20% changed the predicted origin by approximately 1.997 km relative to baseline.
- Increasing current speed by 20% changed the predicted origin by approximately 5.325 km relative to baseline.
- Changing the random seed from 42 to 99 changed the predicted origin by approximately 59.20 m.
- Runtime was measured for all provided runs.
- Real-world predicted-vs-actual origin error remains unavailable because independent ground truth is unavailable.

## Drift Evaluation Final Status

| Test | Status |
|---|---|
| Controlled drift origin error | Measured |
| Provided vs recalculated controlled error | Measured |
| Same-seed reproducibility (seed=42) | PASS |
| Wind sensitivity (+20%) | Measured |
| Current sensitivity (+20%) | Measured |
| Seed variance (42 vs 99) | Measured |
| Runtime | Measured |
| Final uncertainty radius | Measured |
| Hindcast path validity | PASS |
| Backward direction consistency | PASS |
| Uncertainty polygon validity | PASS |
| Predicted origin inside uncertainty polygon | PASS |
| Real predicted-vs-actual origin error | Unavailable |

**Important:** The controlled synthetic scenario validates the drift-engine implementation only. The real SAR scenario provides a probable origin estimate and does not provide verified ground-truth accuracy.

## 5. AIS Quality

Metrics:
- Gap percentage
- Maximum gap duration
- Per-vessel completeness

Source:
`04_ais_track_completeness.ipynb`

Status:
Measured from the available real MarineCadastre AIS sample.

No synthetic AIS data is used.

## 6. Candidate Ranking

Candidate ranking metrics may include:
- candidate position
- estimated travel/drift consistency
- uncertainty containment
- supporting evidence
- ranking score

Current status:
UNAVAILABLE as a validated metric because no independent ground-truth candidate ranking is available.

No candidate ranking score is fabricated for evaluation.

## 7. Metric Availability Summary

| Evaluation Area | Metric | Status | Basis |
|---|---|---|---|
| Segmentation | IoU | Measured | Existing evaluation example |
| Segmentation | Dice | Measured | Existing evaluation example |
| Segmentation | Precision | Measured | Existing evaluation example |
| Segmentation | Recall | Measured | Existing evaluation example |
| Geometry | Area | Measured | Known-value validation |
| Geometry | Centroid | Measured | Known-value validation |
| Geometry | Perimeter | Measured | Known-value validation |
| Drift | Controlled origin error | Measured | Controlled simulation |
| Drift | Real origin error | Unavailable | No ground truth |
| Drift | Hindcast path validity | Available for evaluation | Real SAR output |
| Drift | Uncertainty polygon validity | Available for evaluation | Real SAR output |
| Drift | Uncertainty radius | Measured | Real SAR output |
| Drift | Reproducibility | Available for evaluation | Seed = 42 |
| AIS | Gap percentage | Measured | Real AIS sample |
| AIS | Maximum gap | Measured | Real AIS sample |
| Candidate ranking | Validated ranking accuracy | Unavailable | No independent ground truth |